# baseline v4 - optimized

이 버전은 **정확도와 처리 속도**를 함께 끌어올리기 위해 다음을 반영했습니다.

- 전체 train 데이터 사용
- LoRA + 4bit 양자화 + gradient checkpointing
- 학습 시 **assistant 정답 토큰만 loss 계산**하도록 라벨 마스킹
- 배치 학습/검증/추론
- 검증 정확도 기준 best adapter 저장
- 추론 시 `generate()` 대신 **a/b/c/d next-token scoring** 사용으로 속도 개선
- `pin_memory`, `persistent_workers`, `non_blocking`, `TF32` 등 런타임 최적화


## 1) 환경 준비

아래 셀 실행 후 **런타임 다시 시작** 권장.


In [ ]:
!pip -q install -U   "transformers>=4.52.0"   "accelerate>=1.7.0"   "peft>=0.13.2"   "bitsandbytes==0.46.1"   "datasets>=3.6.0"   "pillow>=11.0.0"   "pandas>=2.2.2"


## 2) 데이터 준비

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
ZIP_PATH = "/content/drive/My Drive/2026-ssafy-15-2-ai.zip"
DATA_ROOT = "/content"

if not (os.path.exists("/content/train.csv") and os.path.exists("/content/test.csv")):
    !unzip -qo "{ZIP_PATH}" -d "{DATA_ROOT}"
else:
    print("압축 해제 생략: 이미 데이터가 존재합니다.")


## 3) 라이브러리 / 설정 / 데이터 로드

In [ ]:
import os, math, random
from dataclasses import dataclass
from typing import Any, Dict, List

import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

Image.MAX_IMAGE_PIXELS = None
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

device = "cuda" if torch.cuda.is_available() else "cpu"
amp_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

print("device:", device)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("amp_dtype:", amp_dtype)

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
IMAGE_SIZE = 448
MAX_NEW_TOKENS = 2

BATCH_SIZE = 2
EVAL_BATCH_SIZE = 4
GRAD_ACCUM = 8
EPOCHS = 2
LR = 2e-4
WARMUP_RATIO = 0.03
NUM_WORKERS = 2 if os.cpu_count() and os.cpu_count() >= 4 else 0

TRAIN_CSV = "/content/train.csv"
TEST_CSV = "/content/test.csv"

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("train:", train_df.shape)
print("test :", test_df.shape)
print(train_df.head(2))


## 4) 모델 / Processor 로드

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=amp_dtype,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=amp_dtype,
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()
base_model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

model_device = next(model.parameters()).device
print("model_device:", model_device)


## 5) 프롬프트 / 데이터셋 / 콜레이터

In [ ]:
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Think carefully about the image and options, then answer with exactly one lowercase letter among a, b, c, or d. "
    "Do not output anything except that one letter."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "정답은 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

def make_messages(row, img, include_answer: bool):
    question = str(row["question"])
    a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
    user_text = build_mc_prompt(question, a, b, c, d)

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": user_text},
        ]},
    ]
    if include_answer:
        gold = str(row["answer"]).strip().lower()
        messages.append({"role": "assistant", "content": [{"type": "text", "text": gold}]})
    return messages

class VQAMCDataset(Dataset):
    def __init__(self, df: pd.DataFrame, train: bool = True):
        self.df = df.reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")

        prompt_messages = make_messages(row, img, include_answer=False)
        full_messages = make_messages(row, img, include_answer=True) if self.train else None

        prompt_text = processor.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        item = {
            "image": img,
            "prompt_text": prompt_text,
        }

        if self.train:
            full_text = processor.apply_chat_template(
                full_messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            item["full_text"] = full_text
            item["answer"] = str(row["answer"]).strip().lower()

        return item

@dataclass
class TrainCollator:
    processor: Any

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        images = [x["image"] for x in batch]
        full_texts = [x["full_text"] for x in batch]
        prompt_texts = [x["prompt_text"] for x in batch]

        enc = self.processor(
            text=full_texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )
        prompt_enc = self.processor(
            text=prompt_texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        labels = enc["input_ids"].clone()
        pad_token_id = self.processor.tokenizer.pad_token_id
        if pad_token_id is not None:
            labels[labels == pad_token_id] = -100

        prompt_lens = prompt_enc["attention_mask"].sum(dim=1).tolist()
        for i, prompt_len in enumerate(prompt_lens):
            labels[i, :prompt_len] = -100

        enc["labels"] = labels
        return enc

@dataclass
class EvalCollator:
    processor: Any

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        images = [x["image"] for x in batch]
        prompt_texts = [x["prompt_text"] for x in batch]

        enc = self.processor(
            text=prompt_texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )
        if "answer" in batch[0]:
            enc["answers"] = [x["answer"] for x in batch]
        return enc


## 6) 데이터 분할 / DataLoader / 선택지 스코어링 함수

In [ ]:
def stratified_split(df: pd.DataFrame, label_col: str = "answer", train_ratio: float = 0.95, seed: int = 42):
    train_parts, valid_parts = [], []
    for _, g in df.groupby(label_col, sort=False):
        g = g.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        split_idx = max(1, int(len(g) * train_ratio))
        if split_idx >= len(g):
            split_idx = len(g) - 1
        train_parts.append(g.iloc[:split_idx])
        valid_parts.append(g.iloc[split_idx:])
    train_part = pd.concat(train_parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    valid_part = pd.concat(valid_parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return train_part, valid_part

train_part, valid_part = stratified_split(train_df, train_ratio=0.95, seed=SEED)
print("train split:", train_part.shape)
print("valid split:", valid_part.shape)

train_ds = VQAMCDataset(train_part, train=True)
valid_ds = VQAMCDataset(valid_part, train=True)
test_ds  = VQAMCDataset(test_df, train=False)

loader_kwargs = dict(
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True
    loader_kwargs["prefetch_factor"] = 2

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=TrainCollator(processor),
    **loader_kwargs,
)
valid_train_loader = DataLoader(
    valid_ds,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=TrainCollator(processor),
    **loader_kwargs,
)
valid_eval_loader = DataLoader(
    valid_ds,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=EvalCollator(processor),
    **loader_kwargs,
)
test_loader = DataLoader(
    test_ds,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=EvalCollator(processor),
    **loader_kwargs,
)

def build_choice_token_map(tokenizer):
    choice_map = {}
    for ch in ["a", "b", "c", "d"]:
        token_ids = set()
        for variant in [ch, ch.upper(), " " + ch, " " + ch.upper()]:
            ids = tokenizer.encode(variant, add_special_tokens=False)
            if len(ids) == 1:
                token_ids.add(ids[0])
        if not token_ids:
            raise ValueError(f"Single-token variant not found for choice: {ch}")
        choice_map[ch] = sorted(token_ids)
    return choice_map

choice_token_map = build_choice_token_map(processor.tokenizer)
choice_list = ["a", "b", "c", "d"]
print(choice_token_map)

def move_batch_to_device(batch: Dict[str, Any], device: torch.device):
    moved = {}
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            moved[k] = v.to(device, non_blocking=True)
        else:
            moved[k] = v
    return moved

@torch.no_grad()
def score_batch_choices(model, batch: Dict[str, Any]) -> List[str]:
    batch = move_batch_to_device(batch, model_device)
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        pixel_values=batch.get("pixel_values"),
        image_grid_thw=batch.get("image_grid_thw"),
    )
    logits = outputs.logits
    last_pos = batch["attention_mask"].sum(dim=1) - 1
    next_token_logits = logits[torch.arange(logits.size(0), device=logits.device), last_pos]

    pred_letters = []
    for row in next_token_logits:
        scores = []
        for ch in choice_list:
            ids = choice_token_map[ch]
            scores.append(row[ids].max().item())
        pred_letters.append(choice_list[int(torch.tensor(scores).argmax().item())])
    return pred_letters

@torch.no_grad()
def evaluate_choice_accuracy(model, loader):
    model.eval()
    total, correct = 0, 0
    for batch in tqdm(loader, desc="Valid accuracy", unit="batch"):
        answers = batch.pop("answers")
        preds = score_batch_choices(model, batch)
        total += len(answers)
        correct += sum(int(p == y) for p, y in zip(preds, answers))
    return correct / max(total, 1)


## 7) 학습

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
num_training_steps = EPOCHS * steps_per_epoch
num_warmup_steps = max(1, int(num_training_steps * WARMUP_RATIO))

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

scaler = torch.cuda.amp.GradScaler(enabled=(amp_dtype == torch.float16))
best_val_acc = -1.0
best_dir = "/content/qwen2_5_vl_3b_lora_best"

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]", unit="batch")
    for step, batch in enumerate(progress_bar, start=1):
        batch = move_batch_to_device(batch, model_device)

        with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=torch.cuda.is_available()):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        running_loss += loss.item()
        if scaler.is_enabled():
            scaler.scale(loss).backward()
        else:
            loss.backward()

        should_step = (step % GRAD_ACCUM == 0) or (step == len(train_loader))
        if should_step:
            if scaler.is_enabled():
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            if scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            progress_bar.set_postfix(loss=f"{running_loss:.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")
            running_loss = 0.0

    model.eval()

    val_loss = 0.0
    val_steps = 0
    with torch.no_grad():
        for batch in tqdm(valid_train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [valid loss]", unit="batch"):
            batch = move_batch_to_device(batch, model_device)
            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=torch.cuda.is_available()):
                outputs = model(**batch)
            val_loss += outputs.loss.item()
            val_steps += 1

    val_loss /= max(val_steps, 1)
    val_acc = evaluate_choice_accuracy(model, valid_eval_loader)
    print(f"[Epoch {epoch+1}] val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained(best_dir)
        processor.save_pretrained(best_dir)
        print(f"Best adapter saved to: {best_dir}")

print("best_val_acc:", best_val_acc)


## 8) best adapter 로드 후 추론

In [ ]:
from peft import PeftModel

base_model.config.use_cache = True

inference_base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=amp_dtype,
    trust_remote_code=True,
)
inference_model = PeftModel.from_pretrained(inference_base, best_dir)
inference_model.eval()

infer_device = next(inference_model.parameters()).device

@torch.no_grad()
def score_batch_choices_infer(model, batch: Dict[str, Any]) -> List[str]:
    moved = {}
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            moved[k] = v.to(infer_device, non_blocking=True)
        else:
            moved[k] = v

    outputs = model(
        input_ids=moved["input_ids"],
        attention_mask=moved["attention_mask"],
        pixel_values=moved.get("pixel_values"),
        image_grid_thw=moved.get("image_grid_thw"),
    )
    logits = outputs.logits
    last_pos = moved["attention_mask"].sum(dim=1) - 1
    next_token_logits = logits[torch.arange(logits.size(0), device=logits.device), last_pos]

    pred_letters = []
    for row in next_token_logits:
        scores = []
        for ch in choice_list:
            ids = choice_token_map[ch]
            scores.append(row[ids].max().item())
        pred_letters.append(choice_list[int(torch.tensor(scores).argmax().item())])
    return pred_letters

preds = []
for batch in tqdm(test_loader, desc="Inference", unit="batch"):
    preds.extend(score_batch_choices_infer(inference_model, batch))

submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": preds,
})
submission_path = "/content/submission.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)
submission.head()


In [ ]:
# 제출 파일 확인
submission["answer"].value_counts()
